# 20 Prepare Schaefer200 Atlas For Brainstorm

## What this notebook does

This notebook merges the earlier atlas-fetch and atlas-text conversion steps into one public-facing stage-2 notebook.

It fetches the Schaefer 2018 200-parcel / 7-network atlas from TemplateFlow, then writes the Brainstorm companion `.txt` label file that must have the same basename as the atlas NIfTI.

## Manuscript linkage

- Main Methods 2.2.2 support
- Supplementary Methods 1.2
- Atlas preparation used by both EEG and BOLD parcel workflows

## Inputs

- a Python environment with `templateflow` and `pandas`
- a readable TemplateFlow cache location (`TEMPLATEFLOW_HOME` or the default cache)

## Outputs written

- the fetched atlas NIfTI in the TemplateFlow cache
- the fetched atlas TSV in the TemplateFlow cache
- a Brainstorm-readable `.txt` label file written next to the atlas NIfTI
- optional copied atlas files in a Brainstorm-visible folder if you set `output_dir`

## Manual dependency

This notebook prepares files for the manual Brainstorm handoff documented in `21_brainstorm_volume_source_and_atlas_import_manual.md` and `docs/manual_steps.md`.


In [ ]:
from pathlib import Path
import os
import shutil

import pandas as pd
from templateflow import api as tflow

# ------------------------------------------------------------------
# User-editable settings
# ------------------------------------------------------------------
atlas_space = "MNI152NLin2009cAsym"
atlas_name = "Schaefer2018"
atlas_desc = "200Parcels7Networks"
atlas_resolution = 1

# Leave as None to keep the atlas only in the TemplateFlow cache.
# Set this to a Windows-visible folder if you want an easy Brainstorm handoff copy.
output_dir = None

# If output_dir already exists, copied files will only be replaced when this is True.
overwrite_copied_files = False

print("TEMPLATEFLOW_HOME:", os.environ.get("TEMPLATEFLOW_HOME", "<default cache>"))


In [ ]:
def _as_single_path(value):
    if isinstance(value, (list, tuple)):
        if len(value) != 1:
            raise ValueError(f"Expected one file, got {len(value)} entries: {value}")
        value = value[0]
    return Path(str(value))


atlas_nii = _as_single_path(
    tflow.get(
        atlas_space,
        resolution=atlas_resolution,
        atlas=atlas_name,
        desc=atlas_desc,
        suffix="dseg",
        extension="nii.gz",
    )
)

atlas_tsv = _as_single_path(
    tflow.get(
        atlas_space,
        atlas=atlas_name,
        desc=atlas_desc,
        suffix="dseg",
        extension="tsv",
    )
)

atlas_txt = atlas_nii.with_suffix("").with_suffix(".txt")

print("Fetched atlas NIfTI:", atlas_nii)
print("Fetched atlas TSV:  ", atlas_tsv)
print("Will write TXT:     ", atlas_txt)


In [ ]:
# ------------------------------------------------------------------
# Convert the TemplateFlow TSV into the Brainstorm label TXT format
# Each line must be: <integer_label> <label_name>
# ------------------------------------------------------------------
labels = pd.read_csv(atlas_tsv, sep="\t")

index_col = None
name_col = None

for column in labels.columns:
    lower = column.lower()
    if index_col is None and lower in {"index", "id", "label", "value"}:
        index_col = column
    if name_col is None and "name" in lower:
        name_col = column

if index_col is None or name_col is None:
    index_col = labels.columns[0]
    name_col = labels.columns[1]

with atlas_txt.open("w", encoding="utf-8") as stream:
    for _, row in labels.iterrows():
        stream.write(f"{int(row[index_col])} {str(row[name_col])}\n")

print("Wrote Brainstorm TXT:", atlas_txt)
print("Columns used:", {"index": index_col, "name": name_col})


In [ ]:
# ------------------------------------------------------------------
# Optional copy to a Brainstorm-visible handoff folder
# ------------------------------------------------------------------
if output_dir is None:
    print("No output_dir set. Using the TemplateFlow cache files in place.")
else:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    files_to_copy = [atlas_nii, atlas_tsv, atlas_txt]
    copied = []

    for source in files_to_copy:
        destination = output_dir / source.name
        if destination.exists() and not overwrite_copied_files:
            print("Keeping existing file:", destination)
        else:
            shutil.copy2(source, destination)
            print("Copied:", destination)
        copied.append(destination)

    print("\nBrainstorm-ready handoff files:")
    for path in copied:
        print(" -", path)


## Next step

After the atlas files are ready, continue with:

- `21_brainstorm_volume_source_and_atlas_import_manual.md`

That manual/hybrid handoff covers Brainstorm source localization, atlas import, and the required outputs for the downstream scout-extraction and parcel-export scripts.
